# Extreme Value Analysis Tutorial

This notebook demonstrates how to use the `ExtremesAnalyzer` class for extreme value analysis, including return period calculations, block maxima extraction, and Peaks Over Threshold (POT) analysis.

## Overview

`ExtremesAnalyzer` is designed to:
- Calculate return periods and return values for extreme events
- Extract block maxima (annual, monthly, seasonal)
- Perform Peaks Over Threshold (POT) analysis with declustering
- Fit extreme value distributions (GEV, Gumbel, Weibull, GPD)
- Visualize return level plots
- Handle flexible time series formats (pandas, datetime64, numeric arrays)

## Setup and Data Generation

Let's start by importing the necessary libraries and generating sample wind speed data with realistic temporal structure.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import magica as ma
from magica.core import DataProcessor, ExtremesAnalyzer

# Set random seed for reproducibility
np.random.seed(42)

# Generate 10 years of daily wind speed data
start_date = '2010-01-01'
end_date = '2019-12-31'
dates = pd.date_range(start=start_date, end=end_date, freq='D')
n_samples = len(dates)

# Base wind speeds (Weibull-like distribution)
base_wind = np.random.weibull(2.5, n_samples) * 6 + 2

# Add seasonal variation (stronger winds in winter)
day_of_year = dates.dayofyear.values
seasonal_factor = 1 + 0.3 * np.cos(2 * np.pi * (day_of_year - 15) / 365)  # Peak in winter
wind_speeds = base_wind * seasonal_factor

# Add some extreme events (storms)
n_storms = 15
storm_indices = np.random.choice(n_samples, n_storms, replace=False)
wind_speeds[storm_indices] += np.random.uniform(10, 20, n_storms)

# Create pandas Series with datetime index
wind_series = pd.Series(wind_speeds, index=dates, name='Wind Speed (m/s)')

print(f"Generated {n_samples} daily wind speed measurements ({len(dates)/365:.1f} years)")
print(f"Date range: {dates[0].date()} to {dates[-1].date()}")
print(f"Min: {wind_speeds.min():.2f} m/s")
print(f"Max: {wind_speeds.max():.2f} m/s")
print(f"Mean: {wind_speeds.mean():.2f} m/s")
print(f"Std: {wind_speeds.std():.2f} m/s")

## Visualizing the Time Series

Let's look at the complete time series and identify extreme events.

In [ ]:
# Plot the complete time series
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Full time series
ax1.plot(dates, wind_speeds, linewidth=0.5, alpha=0.7, color='steelblue')
ax1.axhline(wind_speeds.mean(), color='green', linestyle='--', linewidth=1.5, label='Mean', alpha=0.7)
ax1.axhline(np.percentile(wind_speeds, 95), color='orange', linestyle='--', linewidth=1.5, label='95th percentile', alpha=0.7)
ax1.axhline(np.percentile(wind_speeds, 99), color='red', linestyle='--', linewidth=1.5, label='99th percentile', alpha=0.7)
ax1.set_xlabel('Date')
ax1.set_ylabel('Wind Speed (m/s)')
ax1.set_title('Complete Wind Speed Time Series (10 Years)')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Histogram
ax2.hist(wind_speeds, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.5)
ax2.axvline(wind_speeds.mean(), color='green', linestyle='--', linewidth=2, label='Mean')
ax2.axvline(np.percentile(wind_speeds, 95), color='orange', linestyle='--', linewidth=2, label='95th percentile')
ax2.axvline(np.percentile(wind_speeds, 99), color='red', linestyle='--', linewidth=2, label='99th percentile')
ax2.set_xlabel('Wind Speed (m/s)')
ax2.set_ylabel('Probability Density')
ax2.set_title('Wind Speed Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nExtreme value thresholds:")
print(f"Mean: {wind_speeds.mean():.2f} m/s")
print(f"95th percentile: {np.percentile(wind_speeds, 95):.2f} m/s")
print(f"99th percentile: {np.percentile(wind_speeds, 99):.2f} m/s")
print(f"Maximum: {wind_speeds.max():.2f} m/s")

## Basic Usage: Creating an ExtremesAnalyzer

The `ExtremesAnalyzer` can be created from a `DataProcessor` using the `get_extremes_analyzer()` method. It automatically handles pandas Series with datetime indices.

In [ ]:
# Load data using MagicA's DataProcessor
# The pandas Series with datetime index will be automatically preserved
processor = ma.read_data(wind_series)
print(f"Loaded data: {processor}")

# Create ExtremesAnalyzer (time_unit='years' for return periods in years)
extremes = processor.get_extremes_analyzer(time_unit='years')
print(f"\nCreated ExtremesAnalyzer: {extremes}")

# Check the time span
print(f"Time span: {extremes.time_span:.2f} years")
print(f"Number of observations: {len(extremes.data)}")

## Block Maxima Approach: Generalized Extreme Value (GEV) Distribution

The Block Maxima method extracts the maximum value from fixed time blocks (e.g., annual maxima) and fits a GEV distribution. This is the classical approach in extreme value theory.

In [ ]:
# Extract annual maxima
annual_maxima = extremes.extract_block_maxima(block_size='year', method='max')

print(f"Annual Maxima Analysis:")
print(f"Number of years: {len(annual_maxima)}")
print(f"\nAnnual maximum wind speeds:")
for year, max_wind in annual_maxima.items():
    print(f"  {year}: {max_wind:.2f} m/s")

print(f"\nStatistics of annual maxima:")
maxima_values = list(annual_maxima.values())
print(f"Mean: {np.mean(maxima_values):.2f} m/s")
print(f"Std: {np.std(maxima_values):.2f} m/s")
print(f"Min: {np.min(maxima_values):.2f} m/s")
print(f"Max: {np.max(maxima_values):.2f} m/s")

In [ ]:
# Create a new processor with annual maxima and fit GEV distribution
maxima_processor = ma.read_data(np.array(maxima_values))
maxima_extremes = maxima_processor.get_extremes_analyzer(time_unit='years')

# Fit GEV distribution (genextreme in SciPy)
maxima_extremes.fit_distribution('genextreme')

# Get fitted parameters
params = maxima_extremes.data_processor.get_fitted_params()
print(f"\nFitted GEV Distribution Parameters:")
print(f"Shape (ξ): {params[0]:.4f}")
print(f"Location (μ): {params[1]:.4f}")
print(f"Scale (σ): {params[2]:.4f}")

# Interpret the shape parameter
if params[0] > 0.1:
    print(f"\nInterpretation: Fréchet family (heavy-tailed, bounded below)")
elif params[0] < -0.1:
    print(f"\nInterpretation: Weibull family (short-tailed, bounded above)")
else:
    print(f"\nInterpretation: Gumbel family (exponential tails)")

## Return Period and Return Value Analysis

Now let's calculate return values for different return periods (e.g., 10-year, 50-year, 100-year wind speeds).

In [ ]:
# Calculate return values for various return periods
return_periods = [2, 5, 10, 20, 50, 100]

print("Return Period Analysis (Block Maxima / GEV):")
print("=" * 60)
print(f"{'Return Period':<20} {'Return Value (m/s)':<20} {'Design Use'}")
print("-" * 60)

for T in return_periods:
    rv = maxima_extremes.return_value(T)
    
    # Suggest design use based on return period
    if T <= 5:
        design_use = "Operational limits"
    elif T <= 25:
        design_use = "Standard structures"
    elif T <= 50:
        design_use = "Important structures"
    else:
        design_use = "Critical infrastructure"
    
    print(f"{T} years{'':<13} {rv:.2f}{'':<15} {design_use}")

print("\n💡 Interpretation:")
print("   A 50-year return value has a 2% probability of being exceeded in any given year.")
print("   Over a 50-year period, there's approximately a 63.4% chance of exceeding this value.")

In [ ]:
# Calculate return periods for specific wind speed thresholds
threshold_speeds = [20, 25, 30, 35]

print("\nReturn Period for Specific Wind Speed Thresholds:")
print("=" * 60)
print(f"{'Wind Speed (m/s)':<20} {'Return Period (years)':<25} {'Probability/year'}")
print("-" * 60)

for speed in threshold_speeds:
    T = maxima_extremes.return_period(speed)
    prob = 1.0 / T if T > 0 else 0
    print(f"{speed}{'':<16} {T:.2f}{'':<21} {prob:.4f} ({prob*100:.2f}%)")

print("\n💡 Example: A wind speed of 25 m/s has a return period of X years,")
print("   meaning it's expected to be exceeded once every X years on average.")

## Visualization: Return Level Plot

The return level plot is a fundamental diagnostic tool in extreme value analysis, showing both empirical and theoretical return levels.

In [ ]:
# Create return level plot
fig, ax = plt.subplots(figsize=(12, 7))

maxima_extremes.plot_return_levels(
    ax=ax,
    return_periods=np.logspace(np.log10(1.1), np.log10(200), 100),  # From 1.1 to 200 years
    confidence_level=0.95,
    title='Return Level Plot - Annual Maximum Wind Speeds (GEV Distribution)'
)

# Add reference lines for common design return periods
design_periods = [10, 50, 100]
for T in design_periods:
    rv = maxima_extremes.return_value(T)
    ax.axhline(rv, color='gray', linestyle=':', linewidth=1, alpha=0.5)
    ax.axvline(T, color='gray', linestyle=':', linewidth=1, alpha=0.5)
    ax.text(T * 1.1, rv, f'{T}-year: {rv:.1f} m/s', fontsize=9, 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.show()

print("\n📊 Return Level Plot Interpretation:")
print("   • Blue points: Empirical return levels from observed annual maxima")
print("   • Red line: Theoretical return levels from fitted GEV distribution")
print("   • Shaded area: 95% confidence interval (if calculated)")
print("   • Good agreement indicates the GEV model fits well")
print("   • Deviations at high return periods are common due to extrapolation")

## Peaks Over Threshold (POT) Approach: Generalized Pareto Distribution (GPD)

The POT method identifies all values exceeding a high threshold and fits a GPD to the exceedances. This approach uses more data than block maxima and can be more efficient.

In [ ]:
# Choose threshold (e.g., 95th percentile)
threshold = np.percentile(wind_speeds, 95)
print(f"Threshold for POT analysis: {threshold:.2f} m/s (95th percentile)")

# Extract peaks over threshold with declustering
# min_separation ensures peaks are independent (at least 3 days apart)
peaks, peak_times = extremes.peaks_over_threshold(
    threshold=threshold,
    min_separation=3  # days
)

print(f"\nPeaks Over Threshold Analysis:")
print(f"Number of peaks identified: {len(peaks)}")
print(f"Average exceedance rate: {len(peaks) / extremes.time_span:.2f} peaks/year")
print(f"\nFirst 10 peaks:")
for i, (peak, time) in enumerate(zip(peaks[:10], peak_times[:10])):
    print(f"  {i+1}. {time}: {peak:.2f} m/s (exceedance: {peak - threshold:.2f} m/s)")

In [ ]:
# Visualize the POT extraction
fig, ax = plt.subplots(figsize=(14, 6))

# Plot full time series
ax.plot(dates, wind_speeds, linewidth=0.5, alpha=0.5, color='steelblue', label='All data')

# Highlight threshold
ax.axhline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold ({threshold:.2f} m/s)', alpha=0.7)

# Mark identified peaks
ax.scatter(peak_times, peaks, color='red', s=50, zorder=5, label=f'Peaks (n={len(peaks)})', alpha=0.8)

ax.set_xlabel('Date')
ax.set_ylabel('Wind Speed (m/s)')
ax.set_title('Peaks Over Threshold (POT) Identification with Declustering')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 Declustering ensures peaks are independent (min {3} days apart)")
print(f"   This avoids counting multiple observations from the same storm event.")

In [ ]:
# Fit GPD to exceedances (peaks - threshold)
exceedances = peaks - threshold

exceedance_processor = ma.read_data(exceedances)
pot_extremes = exceedance_processor.get_extremes_analyzer(time_unit='years')

# Fit GPD (genpareto in SciPy)
pot_extremes.fit_distribution('genpareto')

# Get fitted parameters
gpd_params = pot_extremes.data_processor.get_fitted_params()
print(f"Fitted GPD Parameters (for exceedances):")
print(f"Shape (ξ): {gpd_params[0]:.4f}")
print(f"Location: {gpd_params[1]:.4f}")
print(f"Scale (σ): {gpd_params[2]:.4f}")

# Interpret the shape parameter
if gpd_params[0] > 0.1:
    print(f"\nInterpretation: Heavy-tailed distribution (higher probability of extremes)")
elif gpd_params[0] < -0.1:
    print(f"\nInterpretation: Short-tailed distribution (finite upper bound)")
else:
    print(f"\nInterpretation: Exponential-like tails")

## POT Return Value Calculation

For POT analysis, return values must account for the exceedance rate and threshold.

In [ ]:
# Calculate POT-based return values
# Return value = threshold + GPD quantile adjusted for exceedance rate

exceedance_rate = len(peaks) / extremes.time_span  # peaks per year

print("Return Period Analysis (POT / GPD):")
print("=" * 60)
print(f"Exceedance rate: {exceedance_rate:.2f} peaks/year")
print(f"Threshold: {threshold:.2f} m/s\n")
print(f"{'Return Period':<20} {'Return Value (m/s)':<20} {'Exceedance Level'}")
print("-" * 60)

for T in return_periods:
    # For POT, the return period relates to the probability of exceeding the threshold
    # P(X > x | X > u) = (1 - F_GPD(x-u))
    # Return level: x = u + scale * ((λT)^ξ - 1) / ξ, where λ is exceedance rate
    
    # Using the fitted GPD to calculate the quantile
    prob = 1.0 / (exceedance_rate * T)  # Probability in the GPD scale
    if prob < 1.0:
        exceedance_quantile = pot_extremes.data_processor.ppf(1 - prob)
        rv_pot = threshold + exceedance_quantile
    else:
        rv_pot = np.nan
    
    if not np.isnan(rv_pot):
        exceedance = rv_pot - threshold
        print(f"{T} years{'':<13} {rv_pot:.2f}{'':<15} +{exceedance:.2f} m/s")
    else:
        print(f"{T} years{'':<13} {'N/A':<20} {'N/A'}")

print("\n💡 POT return values account for:")
print("   • The threshold level (base wind speed)")
print("   • The exceedance distribution (GPD)")
print("   • The rate of threshold exceedances")

## Comparing Block Maxima vs POT Approaches

Let's compare the return value estimates from both methods.

In [ ]:
# Calculate return values from both methods
comparison_periods = [5, 10, 20, 50, 100]

gev_values = [maxima_extremes.return_value(T) for T in comparison_periods]

# POT return values
pot_values = []
for T in comparison_periods:
    prob = 1.0 / (exceedance_rate * T)
    if prob < 1.0:
        exceedance_quantile = pot_extremes.data_processor.ppf(1 - prob)
        pot_values.append(threshold + exceedance_quantile)
    else:
        pot_values.append(np.nan)

# Comparison table
print("Method Comparison: Block Maxima (GEV) vs POT (GPD)")
print("=" * 80)
print(f"{'Return Period':<15} {'GEV (m/s)':<15} {'GPD (m/s)':<15} {'Difference':<15} {'% Diff'}")
print("-" * 80)

for T, gev_val, pot_val in zip(comparison_periods, gev_values, pot_values):
    if not np.isnan(pot_val):
        diff = pot_val - gev_val
        pct_diff = (diff / gev_val) * 100
        print(f"{T} years{'':<8} {gev_val:<15.2f} {pot_val:<15.2f} {diff:<+15.2f} {pct_diff:+.1f}%")
    else:
        print(f"{T} years{'':<8} {gev_val:<15.2f} {'N/A':<15} {'N/A':<15} {'N/A'}")

print("\n📊 Method Selection Guidelines:")
print("\nBlock Maxima (GEV):")
print("  ✓ More robust, classical approach")
print("  ✓ Less sensitive to threshold selection")
print("  ✓ Better for data with clear seasonal structure")
print("  ✗ Uses less data (only one value per block)")
print("  ✗ Requires long time series for reliability")
print("\nPeaks Over Threshold (GPD):")
print("  ✓ More efficient, uses more extreme data")
print("  ✓ Better for short time series")
print("  ✓ Can adapt to changing extreme behavior")
print("  ✗ Sensitive to threshold choice")
print("  ✗ Requires careful declustering")
print("  ✗ More complex implementation")

## Visualizing Both Approaches Side-by-Side

In [ ]:
# Create comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: GEV return levels
maxima_extremes.plot_return_levels(
    ax=ax1,
    return_periods=np.logspace(np.log10(1.1), np.log10(200), 100),
    confidence_level=0.95,
    title='Block Maxima Approach (GEV)'
)

# Plot 2: Exceedance distribution (GPD)
sorted_exceedances = np.sort(exceedances)
empirical_cdf = np.arange(1, len(sorted_exceedances) + 1) / len(sorted_exceedances)
theoretical_cdf = pot_extremes.data_processor.cdf(sorted_exceedances)

ax2.plot(sorted_exceedances, empirical_cdf, 'bo', markersize=6, alpha=0.6, label='Empirical')
ax2.plot(sorted_exceedances, theoretical_cdf, 'r-', linewidth=2, label='GPD fit')
ax2.set_xlabel('Exceedance (m/s)')
ax2.set_ylabel('Cumulative Probability')
ax2.set_title('POT Approach: GPD Fit to Exceedances')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Advanced: Monthly and Seasonal Block Maxima

Besides annual maxima, we can extract maxima from other time blocks.

In [ ]:
# Extract monthly maxima
monthly_maxima = extremes.extract_block_maxima(block_size='month', method='max')

print(f"Monthly Maxima Analysis:")
print(f"Number of months: {len(monthly_maxima)}")
print(f"Mean monthly maximum: {np.mean(list(monthly_maxima.values())):.2f} m/s")
print(f"\nFirst 12 months:")
for i, (month, max_wind) in enumerate(list(monthly_maxima.items())[:12]):
    print(f"  {month}: {max_wind:.2f} m/s")

In [ ]:
# Visualize seasonal variation in extremes
# Convert monthly maxima to DataFrame for easier analysis
monthly_df = pd.DataFrame([
    {'date': pd.to_datetime(k), 'max_wind': v} 
    for k, v in monthly_maxima.items()
])
monthly_df['month'] = monthly_df['date'].dt.month
monthly_df['month_name'] = monthly_df['date'].dt.strftime('%b')

# Calculate monthly statistics
monthly_stats = monthly_df.groupby(['month', 'month_name'])['max_wind'].agg(['mean', 'std', 'min', 'max']).reset_index()

fig, ax = plt.subplots(figsize=(12, 6))

# Plot mean with error bars
months = monthly_stats['month'].values
means = monthly_stats['mean'].values
stds = monthly_stats['std'].values

ax.plot(months, means, 'o-', linewidth=2, markersize=8, color='steelblue', label='Mean monthly maximum')
ax.fill_between(months, means - stds, means + stds, alpha=0.3, color='steelblue', label='±1 std')
ax.scatter(months, monthly_stats['max'], color='red', s=50, zorder=5, alpha=0.6, label='Absolute maximum')

ax.set_xlabel('Month')
ax.set_ylabel('Wind Speed (m/s)')
ax.set_title('Seasonal Variation in Monthly Maximum Wind Speeds')
ax.set_xticks(months)
ax.set_xticklabels(monthly_stats['month_name'])
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find the most extreme month
most_extreme_month = monthly_stats.loc[monthly_stats['mean'].idxmax()]
print(f"\nMost extreme month on average: {most_extreme_month['month_name']} ({most_extreme_month['mean']:.2f} m/s)")
print(f"Least extreme month on average: {monthly_stats.loc[monthly_stats['mean'].idxmin(), 'month_name']} "
      f"({monthly_stats['mean'].min():.2f} m/s)")

## Goodness-of-Fit Assessment for Extreme Value Distributions

Let's evaluate how well our distributions fit the extreme data.

In [ ]:
# Goodness-of-fit for GEV (annual maxima)
print("Goodness-of-Fit Assessment:")
print("=" * 60)
print("\nBlock Maxima (GEV):")

ks_gev = maxima_extremes.data_processor.goodness_of_fit('ks')
print(f"  Kolmogorov-Smirnov Test:")
print(f"    Statistic: {ks_gev['ks_statistic']:.6f}")
print(f"    P-value: {ks_gev['p_value']:.6f}")
print(f"    Result: {'Good fit ✓' if ks_gev['p_value'] > 0.05 else 'Poor fit ✗'} (α = 0.05)")

rmse_gev = maxima_extremes.data_processor.goodness_of_fit('rmse')
print(f"  RMSE: {rmse_gev:.6f}")

# Goodness-of-fit for GPD (exceedances)
print("\nPeaks Over Threshold (GPD):")

ks_gpd = pot_extremes.data_processor.goodness_of_fit('ks')
print(f"  Kolmogorov-Smirnov Test:")
print(f"    Statistic: {ks_gpd['ks_statistic']:.6f}")
print(f"    P-value: {ks_gpd['p_value']:.6f}")
print(f"    Result: {'Good fit ✓' if ks_gpd['p_value'] > 0.05 else 'Poor fit ✗'} (α = 0.05)")

rmse_gpd = pot_extremes.data_processor.goodness_of_fit('rmse')
print(f"  RMSE: {rmse_gpd:.6f}")

print("\n💡 Note: Small sample sizes in extreme value analysis often result in")
print("   lower statistical power. Visual inspection of Q-Q plots and return")
print("   level plots is equally important.")

## Summary Statistics and Report

Let's generate a comprehensive summary of the extreme value analysis.

In [ ]:
# Get summary statistics from ExtremesAnalyzer
summary = extremes.get_summary_statistics()

print("EXTREME VALUE ANALYSIS SUMMARY REPORT")
print("=" * 80)
print(f"\nDataset Information:")
print(f"  Period: {dates[0].date()} to {dates[-1].date()}")
print(f"  Duration: {extremes.time_span:.2f} years")
print(f"  Total observations: {summary['n_observations']}")
print(f"  Time unit: {summary['time_unit']}")

print(f"\nData Statistics:")
print(f"  Mean: {summary['mean']:.2f} m/s")
print(f"  Std Dev: {summary['std']:.2f} m/s")
print(f"  Minimum: {summary['min']:.2f} m/s")
print(f"  Maximum: {summary['max']:.2f} m/s")
print(f"  95th percentile: {summary['percentile_95']:.2f} m/s")
print(f"  99th percentile: {summary['percentile_99']:.2f} m/s")

if summary['distribution_name']:
    print(f"\nFitted Distribution: {summary['distribution_name']}")
    print(f"  Parameters: {summary['distribution_params']}")

print(f"\nBlock Maxima Analysis:")
print(f"  Annual maxima: {len(annual_maxima)} years")
print(f"  Mean annual maximum: {np.mean(maxima_values):.2f} m/s")
print(f"  GEV shape parameter: {params[0]:.4f}")

print(f"\nPOT Analysis:")
print(f"  Threshold: {threshold:.2f} m/s (95th percentile)")
print(f"  Number of peaks: {len(peaks)}")
print(f"  Exceedance rate: {exceedance_rate:.2f} peaks/year")
print(f"  GPD shape parameter: {gpd_params[0]:.4f}")

print(f"\nDesign Wind Speeds (GEV):")
for T in [10, 50, 100]:
    rv = maxima_extremes.return_value(T)
    print(f"  {T}-year return value: {rv:.2f} m/s")

print("\n" + "=" * 80)

## Best Practices and Recommendations

### When to Use Extreme Value Analysis:

1. **Design of structures** (wind turbines, buildings, bridges)
2. **Risk assessment** (insurance, safety planning)
3. **Climate studies** (return periods of extreme weather)
4. **Ocean engineering** (extreme wave heights)
5. **Hydrology** (flood return periods)

### Key Guidelines:

1. **Data Requirements:**
   - Minimum 10-30 years for reliable annual maxima analysis
   - Data should be stationary (no long-term trends)
   - Quality control is crucial for extreme values

2. **Method Selection:**
   - **Use Block Maxima (GEV)** when:
     - You have long time series (>20 years)
     - Data has strong seasonal patterns
     - Simplicity and robustness are priorities
   
   - **Use POT (GPD)** when:
     - Time series is shorter (<20 years)
     - You want to use more extreme data
     - Extremes don't follow seasonal patterns

3. **Threshold Selection (POT):**
   - Too low: violates GPD assumptions (not in tail)
   - Too high: insufficient data for fitting
   - Start with 90-95th percentile, adjust based on diagnostics

4. **Declustering:**
   - Essential for POT to ensure independence
   - Separation time depends on phenomenon (storms: 3-7 days)
   - Use autocorrelation analysis to guide selection

5. **Return Period Interpretation:**
   - T-year return value ≠ maximum in T years
   - It's the value exceeded on average once every T years
   - Annual exceedance probability = 1/T

6. **Uncertainty:**
   - Extrapolation beyond observed data is uncertain
   - Always report confidence intervals
   - Be especially cautious with return periods > 2 × record length

7. **Model Validation:**
   - Check return level plots visually
   - Compare multiple models (GEV, Gumbel, Weibull)
   - Consider physical plausibility of results

### Common Pitfalls to Avoid:

- ❌ Using insufficient data length
- ❌ Ignoring data quality issues
- ❌ Not checking for trends/non-stationarity
- ❌ Over-extrapolating beyond data range
- ❌ Forgetting to decluster POT peaks
- ❌ Ignoring seasonal effects
- ❌ Not validating model assumptions

### Recommended Workflow:

```python
# 1. Load time series data (pandas Series with datetime index)
processor = ma.read_data(your_series)

# 2. Create extremes analyzer
extremes = processor.get_extremes_analyzer(time_unit='years')

# 3. Block Maxima approach
annual_maxima = extremes.extract_block_maxima(block_size='year')
maxima_processor = ma.read_data(list(annual_maxima.values()))
maxima_extremes = maxima_processor.get_extremes_analyzer(time_unit='years')
maxima_extremes.fit_distribution('genextreme')

# 4. Calculate design values
rv_50 = maxima_extremes.return_value(50)

# 5. Visualize and validate
maxima_extremes.plot_return_levels()

# 6. Compare with POT if needed
peaks, peak_times = extremes.peaks_over_threshold(
    threshold=np.percentile(your_data, 95),
    min_separation=3
)
```

## Further Reading

### Key References:

1. **Coles, S. (2001).** *An Introduction to Statistical Modeling of Extreme Values.*  
   Springer. (The definitive textbook on extreme value theory)

2. **Beirlant, J., et al. (2004).** *Statistics of Extremes: Theory and Applications.*  
   Wiley.

3. **Gumbel, E.J. (1958).** *Statistics of Extremes.*  
   Columbia University Press. (Classic text)

4. **Serinaldi, F. & Kilsby, C.G. (2014).**  
   "Rainfall extremes: Toward reconciliation after the battle of distributions."  
   *Water Resources Research*, 50(1), 336-352.

### Online Resources:

- **R package 'extRemes'**: Excellent documentation and examples
- **Python package 'pyextremes'**: Modern Python implementation
- **NCAR's EVA Tutorial**: Practical guide for climate data

### MagicA Documentation:

- For more on distributions: See the AutoFitter tutorial
- For Monte Carlo analysis: See the Monte Carlo stability tutorial
- API reference: Check the ExtremesAnalyzer documentation